# XGB

In [6]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
48838,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
48839,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
48840,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [4]:
X.select_dtypes(include=["number"]).columns

Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

# 1) 데이터 로드
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.copy()

# 2) 타겟/피처 분리
y = (df["class"] == ">50K").astype(int)   # 1: >50K, 0: <=50K
X = df.drop(columns=["class"])

# 3) 결측 처리 (Adult는 '?' 같은 값이 있을 수 있어 NA로 처리)
X = X.replace("?", pd.NA)
# 수치형은 중앙값, 범주형은 최빈값으로 간단하게 대체
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    X[c] = X[c].fillna(X[c].mode(dropna=True)[0])

# 4) One-Hot 인코딩 (XGBoost는 기본적으로 범주형 raw string을 못 받으니 변환)
X_ohe = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# 5) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=42, stratify=y
)

# 6) 모델 학습
model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

# 7) 평가
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("XGBoost Accuracy:", accuracy_score(y_test, pred))
print("XGBoost ROC-AUC :", roc_auc_score(y_test, proba))

XGBoost Accuracy: 0.8737844201044119
XGBoost ROC-AUC : 0.9301607869099451


# XGB 핵심 하이퍼 파라미터

- n_estimators	boosting 횟수
  - 많을수록 강해짐
- learning_rate (eta)	학습률
  - 작을수록 안정적
- max_depth	트리 깊이
  - 깊으면 과적합
- subsample	row 샘플링 비율
  - 과적합 방지
- colsample_bytree	feature 샘플링 비율
  - 다양성 ↑
- reg_lambda	L2 규제
  - 과적합 방지
- reg_alpha	L1 규제
  - sparse 효과

# 실무 XGB 하이퍼 파라미터 (개인적인 기준)

- n_estimators=500
- learning_rate=0.03~0.1
- max_depth=4~8
- subsample=0.8~0.9
- colsample_bytree=0.8~0.9

In [ ]:
! pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


# Catboost

In [8]:
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

In [9]:
cat_cols

['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

In [10]:
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

In [11]:
cat_idx

[1, 3, 5, 6, 7, 8, 9, 13]

In [12]:
X.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype='object')

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from catboost import CatBoostClassifier

# 1) 데이터 로드
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.copy()

# 2) 타겟/피처 분리
y = (df["class"] == ">50K").astype(int)
X = df.drop(columns=["class"])

# 3) 결측 처리: '?' -> NA (CatBoost는 결측도 다룰 수 있지만, '?'는 문자열이라 NA로 바꿔주는 게 안전)
X = X.replace("?", pd.NA)
# 수치형은 중앙값, 범주형은 최빈값으로 간단하게 대체
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    X[c] = X[c].fillna(X[c].mode(dropna=True)[0])

# 4) cat_features 지정 (전처리 없이 그대로 사용)
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

# 5) train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6) 모델 학습
model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_idx
)

# 7) 평가
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("CatBoost Accuracy:", accuracy_score(y_test, pred))
print("CatBoost ROC-AUC :", roc_auc_score(y_test, proba))

0:	total: 134ms	remaining: 1m 47s
100:	total: 4.95s	remaining: 34.2s
200:	total: 10.8s	remaining: 32.1s
300:	total: 15.7s	remaining: 26.1s
400:	total: 20.5s	remaining: 20.4s
500:	total: 28.1s	remaining: 16.8s
600:	total: 35.3s	remaining: 11.7s
700:	total: 41.4s	remaining: 5.84s
799:	total: 47.1s	remaining: 0us
CatBoost Accuracy: 0.8778790050158665
CatBoost ROC-AUC : 0.9299797659424791


# Cat boost 주요 하이퍼파라미터

- iterations	boosting 횟수
- learning_rate	학습률
- depth	트리 깊이
- l2_leaf_reg	L2 규제
- bagging_temperature	랜덤성 조절

# 실무 Catboost 하이퍼파라미터 (개인적인 기준)
- iterations=1000
- learning_rate=0.03~0.1
- depth=4~8
- l2_leaf_reg=3~10

(애초에 default 값만으로도 성능 괜찮음, 하이퍼파라미터의 영향을 잘 안탐)